In [ ]:
# most recent particle file analyzer

findices = [2, 6] #121 #120 #22 

In [ ]:
# Imports and Folder Selection

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

runs = [TristanRun(findex) for findex in findices]

cc = runs[0].cc

for run in runs:
    print(f"Max step: {np.max(run.steps)}")

my_stepss = [[50], [18]]

In [ ]:
# Config Data

ds = [run.config_data(my_steps, "both") for run, my_steps in zip(runs, my_stepss)]

In [ ]:
print(ds[0].particles)
print(ds[0].particles[1])

In [ ]:
dfs = [d.particles[1].to_dataframe() for d in ds]
#dfs_pos = [d.particles[2].to_dataframe() for d in ds]
q = -1

for df in dfs:
    ann_size(df)
    add_gamma(df)

In [ ]:
#roi = fq_cut(d.fields, 0, [0, run.xlim, 0, run.ylim]) 

In [ ]:
from matplotlib.path import Path

def choose_rect(df, rect):
    [x1, x2, y1, y2] = rect
    return df.loc[(x1 < df['x']) & (df['x'] < x2) & (y1 < df['y']) & (df['y'] < y2)]

In [ ]:
#cses = [[59.5, 60.5, 110, 130], [59.5, 60.5, 161, 170]]
#cses = [[59.5, 60.5, 117, 126], [59.5, 60.5, 161, 170]]
cses = [[59.5, 60.5, 118, 124], [59.5, 60.5, 161, 170]]
#cses = [[59.5, 60.5, 112, 115], [59, 61, 143, 146]]

def get_zoomed_rect(cs_rect):
    [x1, x2, y1, y2] = cs_rect
    y_extend = 10
    x_extent = 10

    xmid = (x1 + x2) / 2

    return [xmid - x_extent, xmid + x_extent, y1 - y_extend, y2 + y_extend]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))

for i in range(2):
    ax = (ax1, ax2)[i]
    roi = fq_cut(ds[i].fields, 0, get_zoomed_rect(cses[i]))

    plot_fq3(roi, roi.dens, plot = ax, vabsmax = 5, cmap = 'viridis', positive = True)
    [x1, x2, y1, y2] = cses[i]
    rect_x = [x1, x2, x2, x1, x1]
    rect_y = [y1, y1, y2, y2, y1]
    ax.plot(rect_x, rect_y, color='red', linewidth=2, label='cses rect')
    ax.set_title(f"Run \# {findices[i]} \n sigma = {runs[i].sigma}, step = {my_stepss[i][0]}")

In [ ]:
sheet_particless = [choose_rect(df, cses[i]) for i, df in enumerate(dfs)]

fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 5))

xaxis, yaxis = ['x', 'w']  # 'x', 'y', 'w', 'p_z', 'gamma'

for i, sheet_prtls in enumerate(sheet_particless):
    ax = (axl, axr)[i]
    ax.scatter(sheet_prtls[xaxis], sheet_prtls[yaxis], s=1, alpha=0.5)
    ax.set_xlabel(xaxis)
    ax.set_ylabel(yaxis)
    ax.set_title(f'Region {i+1} (sigma={runs[i].sigma})')

plt.tight_layout()
plt.show()

In [ ]:
# 2D density plot (hexbin) of $p_z$ vs $y$ for both regions

fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 5))

for i, sheet_prtls in enumerate(sheet_particless):
    ax = (axl, axr)[i]
    hb = ax.hexbin(sheet_prtls['y'], sheet_prtls['w'], gridsize=60, cmap='viridis', bins='log')
    ax.set_xlabel('$y$ [$c / \omega_p$]')
    ax.set_ylabel('$p_z$')
    ax.set_title(f'Region {i+1} (sigma={runs[i].sigma})')
    cbar = fig.colorbar(hb, ax=ax)
    cbar.set_label('log10(N)')

plt.tight_layout()
plt.show()

In [ ]:
def mean_beta(df):
    df['beta_z'] = q * df['w'] / df['gamma']
    return df['beta_z'].mean()

def mean_momentum_beta(df):
    w_mean = df['w'].mean()

    return gamma_to_beta(1 + w_mean ** 2)

def hoshino_scaling(beta):
    beta = np.abs(beta)
    return beta ** 1.5 * np.sqrt(1 - beta ** 2)

repr_betas = [-1, -1]
cs_prtlses = [None, None]

plt.close('all')

for i in range(2):
    ax = (ax1, ax2)[i]
    df = dfs[i]
    rect = cses[i]
    cs_prtlses[i] = choose_rect(dfs[i], cses[i])

    cs_prtls = cs_prtlses[i]

    repr_betas[i] = mean_beta(cs_prtls)
    print(f"Representative beta for sigma = {runs[i].sigma:0.3f}: {repr_betas[i]:0.3f}")
    cs_prtls['beta_z'] = q * cs_prtls['w'] / df['gamma']
    cs_prtls['hs'] = hoshino_scaling(cs_prtls['beta_z'])
    
    # Plot probability density as a line instead of bars
    counts, bins = np.histogram(cs_prtls['beta_z'], bins=25, density=True)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    plt.plot(bin_centers, counts, label=f'sigma={runs[i].sigma}') #num_prlts = {len(cs_prtls)}
    plt.xlabel('beta_z')
    plt.ylabel('Density')
    plt.title('Histogram of beta_z')
    plt.legend()

plt.show()

print(dfs[1]['hs'].mean() / dfs[0]['hs'].mean())

predicted_tearing_ratio = hoshino_scaling(repr_betas[1]) / hoshino_scaling(repr_betas[0])

measured_val = 4
print(f"Predicted tearing ratio: {predicted_tearing_ratio:0.3f} vs {measured_val}")

In [ ]:
mean_densities = []
for i in range(2):
    rect = cses[i]
    roi = fq_cut(ds[i].fields, 0, rect)
    mean_density = roi.dens.mean().compute().item()
    print(f"Mean density in rectangle for sigma={runs[i].sigma}: {mean_density:.4f}")

In [ ]:
mean_pressures = [-1, -1]
outside_B0s = [0.45, 0.25]

for i in range(2):
    csp, run = cs_prtlses[i], runs[i]

    conv_factor = 2 / (run.sigma * run.ppc)

    csp['beta_x'] = csp['u'] / csp['gamma']
    csp['px'] = csp['u'] * csp['beta_x'] * conv_factor * run.ppc / outside_B0s[i] ** 2 # normalized pressure if cs has inflow density relative to outside
    mean_pressures[i] = csp['px'].mean()
    print(f"Mean px for cs rect for sigma = {run.sigma}: {mean_pressures[i]:0.5f}")

ratio = mean_pressures[0] / mean_pressures[1]
print(f"Pressure ratio: {ratio:0.3f}")

In [ ]:
print 

In [ ]:
print(len(chosen_sets[0]))

In [ ]:
xdiff = 100
crect = [run.xlim / 2 - xdiff, run.xlim / 2 + xdiff, 0, run.ylim / 2]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (10, 5))
fig.suptitle(run.boilerplate + f"\n t = {my_steps[0]}")

# Left Plot

for i in range(len(chosen_sets)):
    chosen = chosen_sets[i]
    alphas = [1, 1, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05]
    ax1.scatter(chosen['x'], chosen['y'], s = 0.1, c = colors[i] , alpha = alphas[i])


plot_streamline(ax1, roi, roi.bx, roi.by, 0.75, 'w')
mesh = plot_fq3(roi, roi.dens, plot = ax1, vabsmax = 4)
#ax1.contour(cs.x, cs.y, chi, [0], colors = 'green')

bthresh = 0.9
ax1.contour(roi.x, roi.y, roi.pb, [bthresh], colors = 'green')

ax1.set_xlim(crect[0], crect[1])
ax1.set_ylim(crect[2], crect[3])
ax1.set_aspect('equal')

cbar = fig.colorbar(mesh, ax = ax1)
cbar.set_label("$n / n_0$")

# Right Plot

log_bins = 10 ** np.linspace(-4, 4, 100) #300

for i in range(len(chosen_sets)):
    chosen = chosen_sets[i]
    add_gamma(chosen)
    chosen['beta_w'] = chosen['w'] / chosen['gamma']

    hist, bins = np.histogram((chosen['gamma'] - 1) / run.sigma, bins = log_bins, density = True)
    
    mean = np.mean(chosen['beta_w'])

    ax2.loglog(bins[1:], hist, color = colors[i], label = fr'{len(chosen):.2e} prtls ($\beta_w =$ {mean:.2f})')


ax2.set_xlabel("$(\gamma - 1) / \sigma$")
ax2.set_ylabel("$dN / d\gamma$")
ax2.set_title(f"Electron Spectra")
ax2.grid(visible = True)
ax2.axvline(1, color = 'black', linestyle = '--', label = '$\gamma - 1 = \sigma$')
ax2.axvline(1 / run.sigma, color = 'orange', linestyle = '--', label = '$\gamma - 1 = 1$')
#ax2.legend()
plt.tight_layout()

In [ ]:
# Eckart Frame

total_rmass = np.sum(chosen['gamma'])

bulk_u = np.sum(chosen['u']) / total_rmass
bulk_v = np.sum(chosen['v']) / total_rmass
bulk_w = np.sum(chosen['w']) / total_rmass 

beta_A = run.beta_A

print(f"bulk u / v_A = {bulk_u / beta_A}, bulk v / v_A = {bulk_v / run.beta_A}, bulk w / v_A = {bulk_w / run.beta_A}")

In [ ]:
def g2b(gamma):
    return np.sqrt(1 - 1 / gamma ** 2)

def b2g(beta):
    return 1 / np.sqrt(1 - beta ** 2)

def fourvel2beta(u):
    return u / np.sqrt(1 + u ** 2)


In [ ]:
# COM Frame (warning overwrites the above)

total_rmass = np.sum(chosen['gamma'])

bulk_u = np.sum(chosen['u']) / total_rmass
bulk_v = np.sum(chosen['v']) / total_rmass
bulk_w = np.sum(chosen['w']) / total_rmass 

v_A = run.beta_A / (1 - run.beta_A ** 2) ** 0.5

print(f"bulk u / v_A = {bulk_u / v_A}, bulk v / v_A = {bulk_v / v_A}, bulk w / v_A = {bulk_w / v_A}")

In [ ]:
from scipy.stats import norm

fig, ax = plt.subplots(figsize=(10, 5))

# Plot histogram as lines
hist_u, bins_u = np.histogram(chosen['u'], bins=50, density=True)
hist_w, bins_w = np.histogram(chosen['w'], bins=50, density=True)
bin_centers_u = (bins_u[:-1] + bins_u[1:]) / 2
bin_centers_w = (bins_w[:-1] + bins_w[1:]) / 2

ax.plot(bin_centers_u, hist_u, label='u', color='blue')
ax.plot(bin_centers_w, hist_w, label='w', color='red')

# Fit Gaussian for 'u'
mu_u, std_u = norm.fit(chosen['u'])
p_u = norm.pdf(bin_centers_u, mu_u, std_u)
ax.plot(bin_centers_u, p_u, 'b--', label=f'Fit u: $\mu={mu_u:.2f}$, $\sigma={std_u:.2f}$')

# Fit Gaussian for 'w'
mu_w, std_w = norm.fit(chosen['w'])
p_w = norm.pdf(bin_centers_w, mu_w, std_w)
ax.plot(bin_centers_w, p_w, 'r--', label=f'Fit w: $\mu={mu_w:.2f}$, $\sigma={std_w:.2f}$')

ax.set_xlabel('Value')
ax.set_ylabel('Frequency')
ax.set_title('Histogram of u and w with Gaussian Fits')
ax.legend()
plt.show()

In [ ]:
def y_lorentz_boost(beta_y_frame, df):
    df['beta_x'] = df['u'] / df['gamma']
    df['beta_y'] = df['v'] / df['gamma']
    df['beta_z'] = df['w'] / df['gamma']

    bulk_gamma = b2g(beta_y_frame)

    df['denom'] = (1 - beta_y_frame * df['beta_y'])

    df['beta_xp'] = (df['beta_x'] / bulk_gamma) / df['denom']
    df['beta_yp'] = (df['beta_y'] / bulk_gamma - beta_y_frame) / df['denom']
    df['beta_zp'] = (df['beta_z'] / bulk_gamma) / df['denom']

    df['gammap'] = 1 / np.sqrt(1 - df['beta_xp'] ** 2 - df['beta_yp'] ** 2 - df['beta_zp'] ** 2)

    df['up'] = df['beta_xp'] * df['gammap']
    df['vp'] = df['beta_yp'] * df['gammap']
    df['wp'] = df['beta_zp'] * df['gammap']
    
    return df

In [ ]:
# Frame Transformation

#chosen = y_lorentz_boost(-run.beta_A, chosen)
chosen = y_lorentz_boost(fourvel2beta(bulk_v), chosen)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bins = 50

# Calculate histogram and Gaussian fit for 'up'
hist_up, bins_up = np.histogram(chosen['up'], bins=bins, density=True)
bin_centers_up = (bins_up[:-1] + bins_up[1:]) / 2
mu_up, std_up = norm.fit(chosen['up'])
p_up = norm.pdf(bin_centers_up, mu_up, std_up)

# Plot histogram and Gaussian fit for 'up'
ax.plot(bin_centers_up, hist_up, label='$u$\'', color='blue')
ax.plot(bin_centers_up, p_up, 'b--', label=f'Fit up: $\mu={mu_up:.2f}$, $\sigma={std_up:.2f}$')

# Calculate histogram and Gaussian fit for 'vp'
hist_vp, bins_vp = np.histogram(chosen['vp'], bins=bins, density=True)
bin_centers_vp = (bins_vp[:-1] + bins_vp[1:]) / 2
mu_vp, std_vp = norm.fit(chosen['vp'])
p_vp = norm.pdf(bin_centers_vp, mu_vp, std_vp)

# Plot histogram and Gaussian fit for 'vp'
ax.plot(bin_centers_vp, hist_vp, label='$v$\'', color='green')
ax.plot(bin_centers_vp, p_vp, 'g--', label=f'Fit vp: $\mu={mu_vp:.2f}$, $\sigma={std_vp:.2f}$')

# Calculate histogram and Gaussian fit for 'wp'
hist_wp, bins_wp = np.histogram(chosen['wp'], bins=bins, density=True)
bin_centers_wp = (bins_wp[:-1] + bins_wp[1:]) / 2
mu_wp, std_wp = norm.fit(chosen['wp'])
p_wp = norm.pdf(bin_centers_wp, mu_wp, std_wp)

# Plot histogram and Gaussian fit for 'wp'
ax.plot(bin_centers_wp, hist_wp, label='$w$\'', color='red')
ax.plot(bin_centers_wp, p_wp, 'r--', label=f'Fit wp: $\mu={mu_wp:.2f}$, $\sigma={std_wp:.2f}$')

ax.set_xlabel('Value')
ax.set_ylabel('Frequency')
ax.set_title('Histograms of vp, up, and wp with Gaussian Fits')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
print(2 + 2)

In [ ]:
from scipy.optimize import curve_fit

# Define your arbitrary function

def maxwell_juttner(gamma, coeff, theta):
    return coeff * gamma ** 2 * g2b(gamma) * np.exp(-gamma / theta)

fig, ax = plt.subplots(figsize=(10, 5))

# Plot histogram as lines
hist_gammap, bins_gammap = np.histogram(chosen['gammap'], bins=50, density=True)
bin_centers_gammap = (bins_gammap[:-1] + bins_gammap[1:]) / 2

ax.loglog(bin_centers_gammap - 1, hist_gammap, label='simulation spectrum', color='purple')

# Fit arbitrary function to 'gammap'
popt, pcov = curve_fit(maxwell_juttner, bin_centers_gammap, hist_gammap)
p_gammap = maxwell_juttner(bin_centers_gammap, *popt)
ax.loglog(bin_centers_gammap - 1, p_gammap, '--', color='purple', label=fr'$T / mc^2$ = {popt[1]:.2f} Maxwell-Jüttner')

ax.set_xlabel('$\gamma\' - 1$')
ax.set_ylabel('$f(\gamma\')$')
ax.legend()
ax.grid()

save_polished_fig("mjf")

plt.show()

In [ ]:
print(run.xlim)

In [ ]:
# Calculate the pitch angle and energy for each particle
chosen['v_mag'] = np.sqrt(chosen['u']**2 + chosen['v']**2 + chosen['w']**2)
chosen['B_mag'] = np.sqrt(chosen['bx']**2 + chosen['by']**2 + chosen['bz']**2)
chosen['v_dot_B'] = chosen['u']*chosen['bx'] + chosen['v']*chosen['by'] + chosen['w']*chosen['bz']
chosen['pitch_angle'] = np.degrees(np.arccos(chosen['v_dot_B'] / (chosen['v_mag'] * chosen['B_mag'])))
chosen['energy'] = chosen['gamma'] - 1

# Convert pitch angle to radians for polar plot
chosen['pitch_angle_rad'] = np.radians(chosen['pitch_angle'])

# Compute 2D histogram for pitch angle and energy with improved resolution
hist, xedges, yedges = np.histogram2d(chosen['pitch_angle'], chosen['energy'], bins=[100, 100], density=True)

# Convert pitch angle to radians for polar plot
chosen['pitch_angle_rad'] = np.radians(chosen['pitch_angle'])

# Plot the 2D histogram as a polar density plot
fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(10, 5))
theta, r = np.meshgrid(np.radians(xedges), yedges)
pcm = ax.pcolormesh(theta, r, hist.T, cmap='viridis')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Density')

ax.set_xlabel('Pitch Angle')
ax.set_ylabel('$\gamma - 1$')
ax.set_title('Density of Pitch Angle vs Energy')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate sin^2 of the pitch angle
chosen['sin2_pitch_angle'] = np.sin(chosen['pitch_angle_rad']) ** 2

# Compute 2D histogram for sin^2(pitch angle) and gamma
hist, xedges, yedges = np.histogram2d(chosen['sin2_pitch_angle'], chosen['gamma'], bins=[100, 100], density=True)

# Plot the 2D histogram as a density plot
fig, ax = plt.subplots(figsize=(10, 5))
pcm = ax.pcolormesh(xedges, yedges, hist.T, cmap='viridis')
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Density')

ax.set_xlabel('sin2(Pitch Angle)')
ax.set_ylabel('Gamma')
ax.set_title('Density of sin2(Pitch Angle) vs Gamma')

plt.tight_layout()
plt.show()